In [1]:
from google.colab import files
import zipfile
import os

# Colab에서 업로드
uploaded = files.upload()

# 압축 해제
with zipfile.ZipFile("model.zip", 'r') as zip_ref:
    zip_ref.extractall("custom_model")  # 원하는 경로로 지정


Saving model.zip to model.zip


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

tokenizer = AutoTokenizer.from_pretrained("./custom_model")
model = AutoModelForSequenceClassification.from_pretrained("./custom_model")

clf_pipeline = pipeline("text-classification", model=model, tokenizer=tokenizer)


Device set to use cuda:0


In [3]:
from transformers import pipeline

pipe = pipeline("text-classification", model=model, tokenizer=tokenizer)

pipe("이따위로 일할 거면 그만두세요.")


Device set to use cuda:0


[{'label': 'LABEL_1', 'score': 0.9949126243591309}]

In [4]:
from google.colab import files
import pandas as pd

# 1. 파일 업로드
qa_file = files.upload()

# 2. 파일명 추출
filename = next(iter(qa_file))

# 3. 인코딩 명시해서 CSV 불러오기
df = pd.read_csv(filename, encoding='cp949')  # 또는 'euc-kr'
df = df[df["sentence"].notna()]
df["sentence"] = df["sentence"].astype(str)


Saving user_sentences_labeled.csv to user_sentences_labeled.csv


In [5]:
# 예측
predictions = clf_pipeline(df["sentence"].tolist(), truncation=True)

# 결과 붙이기
df["pred_label"] = [p["label"] for p in predictions]

# 결과 확인
df.head()


,sentence,label,pred_label
0,재결재오류가 반복이되어요,0,LABEL_0
1,유효한 카드를 등록해서 결재진행을,0,LABEL_0
2,하려는데 왜 계속 안되는거죠?,0,LABEL_0
3,38분 사용 0원아닌가요?,0,LABEL_0
4,1000원결재해야되는건가요?,0,LABEL_0


In [6]:
# UTF-8로 저장
utf8_filename = "user_sentences_utf8.csv"
df.to_csv(utf8_filename, index=False, encoding='utf-8-sig')

# 다운로드 버튼 생성
from google.colab import files
files.download(utf8_filename)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 틀린 문장 찾기

In [ ]:
import pandas as pd
from google.colab import files

# 1. 파일 업로드
uploaded = files.upload()

# 2. 파일명 추출
filename = next(iter(uploaded))

# 3. CSV 불러오기
df = pd.read_csv(filename, encoding='utf-8')

# 4. 결측치 제거 및 문자열 처리
df = df[df["sentence"].notna()]
df["sentence"] = df["sentence"].astype(str)

# 5. 예측 틀린 문장만 추출
wrong_df = df[df["label"] != df["pred_label"]]

# 6. 결과 저장
output_filename = "prediction_errors.csv"
wrong_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

# 7. 다운로드
files.download(output_filename)


Saving user_sentences_utf8 (1).csv to user_sentences_utf8 (1) (1).csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 예측하기

In [ ]:
from sklearn.metrics import f1_score, classification_report

# 예측값 후처리: label_0 → 0
df["pred_int"] = df["pred_label"].apply(lambda x: int(x.split("_")[-1]))

# 정답값도 정수형으로
y_true = df["label"].astype(int)
y_pred = df["pred_int"]

# F1 점수 계산
f1 = f1_score(y_true, y_pred, average='weighted')
print(f"F1 Score (weighted): {f1:.4f}")

# 분류 리포트 출력
print(classification_report(y_true, y_pred))


F1 Score (weighted): 0.9068
              precision    recall  f1-score   support

           0       0.98      0.90      0.94      1884
           1       0.45      0.85      0.59       188

    accuracy                           0.89      2072
   macro avg       0.72      0.87      0.76      2072
weighted avg       0.94      0.89      0.91      2072

